## About this project

`ai-minerals` is a portfolio demonstration of **mineral prospectivity mapping
(MPM)** — the use of machine learning on public geoscientific data to rank
locations by their likelihood of hosting a mineral deposit. The target
audience is AI-exploration companies like [KoBold Metals](https://koboldmetals.com/),
[ExploreTech](https://exploretech.ai/), and [Earth AI](https://earth-ai.com/),
who are building production-grade versions of exactly this kind of pipeline.

The repo covers two regions with the same infrastructure:

- **v1: Eastern Alaska**, porphyry Cu-Mo-Au-Ag belt (this notebook's region).
- **v2: BC Golden Triangle**, multi-deposit-class model (porphyry + epithermal +
  skarn + VMS) with a distribution-based blind test on 366 post-2015 drill
  holes. See [`notebooks/bcgt/`](../bcgt/).

All code, data sources, and model outputs are public. The writeup — not the
code — is the primary artifact; this notebook is the entry point for the
Eastern Alaska track.

## About this notebook

This notebook is the **hands-on tour** of the Eastern Alaska datasets.
Everything else in the `notebooks/eastak/` series depends on the data
acquired here; the pipeline is:

1. **This notebook** — walk through each raw data source, verify it covers
   the AOI, build intuition for what the signals look like.
2. **[`baseline_model.qmd`](baseline_model.qmd)** — feature pipeline + logistic
   regression with spatial block cross-validation.
3. **[`random_forest_and_shap.qmd`](random_forest_and_shap.qmd)** — Random
   Forest + SHAP interpretation; diagnoses the exploration-density confound.
4. **[`validation.qmd`](validation.qmd)** — sensitivity pass (strict labels,
   PU baseline, no-geochem ablation, `*_has_data` indicator test).
5. **[`eastak_porphyry_prospectivity.qmd`](eastak_porphyry_prospectivity.qmd)** —
   integrated report rolling all four together.

Each section ends with an "interpretation" note explaining what the output
is telling you about the exploration problem — not just the data.

## About the region

**Eastern Alaska** here means three contiguous USGS 1:250,000-scale
quadrangles covering ~62,000 km² of east-central Alaska:

- **Tanacross (TC)** — 63–64 °N, 143–141 °W — Yukon-Tanana upland
- **Mt. Hayes (MH)** — 63–64 °N, 147–144 °W — Wrangellia / Delta Range
- **Nabesna (NB)** — 62–63 °N, 144–141 °W — Wrangellia / eastern Alaska Range

This AOI straddles the **Wrangellia–Yukon-Tanana tectonic boundary** — the
regional framework within which KoBold's Skolai Ni-Cu-Co-PGM project
sits. It's an established porphyry belt with 56 porphyry-family
occurrences in the USGS Alaska Resource Data File (ARDF) and several
active greenfield exploration programs.

## The target deposit class

**Porphyry Cu-Mo-Au-Ag** deposits form from shallow-level felsic-to-
intermediate intrusions associated with convergent-margin magmatism. The
ore forms in a stockwork of fractures above and around the intrusion, with
characteristic metal zonation out from the central intrusive core: Cu-Mo
in the core, Cu-Au in the inner halo, and Ag/As/Sb/Te in the distal halo.
That zonation is *what our features are trying to detect*.

## The datasets

Nine layers. Each has its own section below; the short version is:

| Layer | Source | Role |
|---|---|---|
| **ARDF** (occurrence labels) | USGS Alaska Resource Data File | 56 porphyry-family positives — the *what we're trying to find* |
| **MRDS** (occurrence mask) | USGS Mineral Resources Data System | any-commodity exclusion for pseudo-negatives |
| **AGDB4** (geochemistry) | USGS Alaska Geochemical Database v4 | stream-sediment + soil pathfinder elements |
| **SGMC** (bedrock geology) | USGS Alaska state geology compilation | lithology one-hot + fault lines |
| **NRCan 100 m aeromagnetic** (residual total field) | Natural Resources Canada Alaska-Yukon compilation | magnetic anomaly map |
| **NRCan 100 m aeromagnetic** (1st vertical derivative) | same | edge-sharpening processed product |
| **USGS Bouguer gravity** | USGS national composite | density anomaly |
| **Copernicus GLO-30 DEM** | Copernicus / ESA | elevation, slope, TRI |
| **Sentinel-2 L2A** | ESA + Microsoft Planetary Computer | alteration-index proxies via SWIR ratios |

Every dataset is open and public-domain. No API keys, no gated downloads.
Raw downloads go to `data/raw/<dataset>/`; the feature pipeline
([`src/ai_minerals/features/assemble.py`](../../src/ai_minerals/features/assemble.py))
glues them into a single pixel-level DataFrame for modeling.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rioxarray

from ai_minerals.aoi import EASTERN_ALASKA, WORKING_CRS
from ai_minerals.data._common import DATA_RAW

AOI = EASTERN_ALASKA
AOI_KEY = AOI.name.lower()  # 'eastak' — used in filenames

print(f"AOI: {AOI.name}")
print(f"  bounding box: W={AOI.min_lon}, S={AOI.min_lat}, "
      f"E={AOI.max_lon}, N={AOI.max_lat}  (WGS84)")
print(f"Working projection: {WORKING_CRS}  (NAD83 / Alaska Albers)")

## 1. The exploration question

We have a **2° latitude × 6° longitude** patch of eastern Alaska — roughly
**222 × 300 km** on the ground — covering three contiguous 1:250,000
quadrangles (Tanacross, Mt Hayes, Nabesna) that span the
Wrangellia–Yukon-Tanana tectonic boundary. We want to build a model that
says "**this pixel is more likely to host a porphyry copper deposit than
that pixel**." That's mineral prospectivity mapping.

A porphyry copper system is a specific *kind* of deposit. The short version:
a buried granitic intrusion cools and releases hot, metal-rich fluids; those
fluids alter the surrounding rock and deposit copper-molybdenum-gold sulfide
minerals in a roughly bullseye pattern around the intrusion. Porphyries leave
several signatures we can try to detect from a distance:

1. **Geochemical halos** — anomalously high Cu, Mo, Au, As, Sb, Bi in rocks
   and stream sediments drained from the alteration zone.
2. **Hydrothermal alteration** — clay minerals, iron oxides, and sericite
   that appear in satellite imagery at specific wavelengths.
3. **Geophysical anomalies** — the granitic intrusion is often magnetic and
   slightly less dense than the country rock.
4. **Structural controls** — porphyries sit on regional-scale faults that
   provided fluid pathways.

The eight datasets we pulled cover each of these signatures plus the
labels (where porphyries are already known). Below, each in turn.

## 2. Known deposits — the labels (MRDS + ARDF)

In [ ]:
# Load MRDS (global) and ARDF (Alaska-specific) occurrences.
mrds_path = DATA_RAW / "mrds" / f"mrds_{AOI_KEY}.geojson"
mrds = gpd.read_file(mrds_path)

ardf_path = DATA_RAW / "ardf" / f"ardf_{AOI_KEY}.gpkg"
ardf = gpd.read_file(ardf_path)

print(f"MRDS: {len(mrds)} occurrences in bbox")
print(f"ARDF: {len(ardf)} occurrences across TC + MH + NB quadrangles (AOI-clipped)")

MRDS and ARDF both describe the same universe of sites (they cross-reference
each other) but from different angles. MRDS is the global USGS occurrence
database; ARDF is Alaska-specific with better-curated deposit-model labels.
For this region ARDF is the cleaner of the two.

In [ ]:
print("ARDF deposit-model distribution (top 10):")
print(ardf["dep_model"].fillna("(none)").value_counts().head(10).to_string())

Notice the geologic vocabulary: "**Porphyry Cu-Mo (Cox and Singer model
21a)**", "**Tintina Belt-style gold-arsenopyrite-quartz vein**", "**Gold-quartz
in veins and brecciated zones**". These are *deposit-model codes* —
compressed geological theories about how each kind of deposit forms. Cox &
Singer (1986) published the canonical list; the model numbers (21a = porphyry
Cu, 22c = low-sulfidation epithermal gold, 39a = placer gold, etc.) are
standard vocabulary across the industry.

Eastern Alaska hosts three kinds of systems simultaneously, all well
represented in our three-quadrangle AOI:

- Cretaceous-age **porphyry Cu-Mo** (Taurus Cu-Mo-Au-Ag in Tanacross is
  the best-studied; Mt Hayes and Nabesna have their own cluster on the
  Wrangellia side of the terrane boundary).
- Younger **orogenic gold** veins (Tintina Belt-style).
- Quaternary **placer gold** downstream of both of the above.

For v1 we're targeting *porphyry Cu-Mo only*. We'll use the orogenic gold
and placer records as part of the "not porphyry" landscape, and we'll check
later whether the model distinguishes the two.

In [ ]:
# Positive-class definition: use Cox & Singer deposit-model codes, not the
# free-text `dep_model` string. The code field is structured and drops the
# noisiest records (speculative "porphyry?" with no code, or "Cu skarn").
#
# Two definitions, side by side:
#   - PORPHYRY FAMILY  (primary)  — all intrusion-hosted porphyry types
#       17  : Porphyry Cu (plutonic, low-F)
#       20c : Porphyry Cu, skarn-related
#       21a : Porphyry Cu-Mo  (canonical)
#       21b : Porphyry Mo, low-F
#   - STRICT 21a (sensitivity check) — only the canonical Porphyry Cu-Mo.
#
# A sensitivity comparison runs after the validation notebook: report whether
# AUC moves meaningfully between family (N~56) and strict (N~32) positives.

import re

PORPHYRY_FAMILY_CODES = ("17", "20c", "21a", "21b")
PORPHYRY_STRICT_CODES = ("21a",)

def _matches_codes(series: pd.Series, codes: tuple[str, ...]) -> pd.Series:
    pat = r"\b(?:" + "|".join(re.escape(c) for c in codes) + r")\b"
    return series.fillna("").str.contains(pat, case=False, regex=True)

family_mask = _matches_codes(ardf["model_code"], PORPHYRY_FAMILY_CODES)
strict_mask = _matches_codes(ardf["model_code"], PORPHYRY_STRICT_CODES)

ardf_porphyry = ardf[family_mask].copy()          # primary
ardf_porphyry_strict = ardf[strict_mask].copy()   # sensitivity check

print(f"Porphyry family   (17/20c/21a/21b):  {len(ardf_porphyry)} records")
print(f"Porphyry strict   (21a only):        {len(ardf_porphyry_strict)} records")

print("\nmodel_code distribution in the family set:")
print(ardf_porphyry["model_code"].fillna("(none)").value_counts().to_string())

These **~56 porphyry-family records** are our primary training positives,
up from 15 in Tanacross alone — the main reason we expanded the AOI. The
codes cover the canonical porphyry Cu-Mo (21a) plus three genetically
related sub-types:

- **Model 17** — plutonic, low-fluorine porphyry Cu. Same deposit
  mechanism, slightly different geologic setting.
- **Model 20c** — porphyry Cu where the skarn carapace is the dominant
  manifestation. Same intrusion, same fluids, skarn ore instead of
  disseminated porphyry.
- **Model 21b** — porphyry Mo, low-F. Mo-dominant variant of 21a.

Records coded as pure Cu skarn (18a) or polymetallic veins (22c) without
an accompanying porphyry interpretation are excluded; so are the 5
records with no model code (mostly speculative "porphyry?" attributions).

Still small in absolute terms — class imbalance is a defining feature of
mineral prospectivity mapping: out of millions of pixels in the AOI,
only a few dozen are confirmed deposits. The [baseline-model notebook](baseline_model.qmd) addresses this
directly (pseudo-negative sampling); the
[validation notebook](validation.qmd) adds the PU-learning comparison.
Beyond that we'll
retrain on the strict 21a-only set and report whether model performance
moves meaningfully between the two positive definitions.

## 3. Geochemistry (AGDB4)

In [ ]:
agdb_path = DATA_RAW / "agdb4" / f"agdb4_samples_{AOI_KEY}.parquet"
agdb = pd.read_parquet(agdb_path)
print(f"AGDB4 samples in AOI: {len(agdb):,}")

print("\nSample source types (how the sample was collected):")
print(agdb["SAMPLE_SOURCE"].fillna("(none)").value_counts().head(6).to_string())

print("\nSample primary class (what was tested):")
print(agdb["PRIMARY_CLASS"].fillna("(none)").value_counts().head(5).to_string())

~24,600 samples in our AOI — a surprisingly dense geochemical record
for a ~222 × 300 km patch of Alaska (about one sample per ~3 km²). A bit
over ⅓ are stream-sediment samples (sediment collected where creeks
drain the landscape — each sample *integrates* rock fingerprinted from
the entire upstream catchment). That's huge for exploration: one creek
can tell you about square kilometers of otherwise inaccessible terrain.

Each sample has been analyzed by one or more methods for dozens of elements.
We pulled the *location* table (`Geol_DeDuped.txt`); the best-value element
data lives in parallel `BV_*.txt` tables that we'll join in the feature-pipeline step.

Let's see where the samples are and what agencies collected them.

In [ ]:
# Plot sample locations colored by agency.
fig, ax = plt.subplots(figsize=(10, 5))
agdb_gdf = gpd.GeoDataFrame(
    agdb, geometry=gpd.points_from_xy(agdb.LONGITUDE, agdb.LATITUDE), crs="EPSG:4326"
).to_crs(WORKING_CRS)

top_agencies = agdb["AGENCY"].value_counts().head(4).index.tolist()
for agency in top_agencies:
    sub = agdb_gdf[agdb_gdf["AGENCY"] == agency]
    ax.scatter(sub.geometry.x, sub.geometry.y, s=2, alpha=0.5, label=f"{agency} ({len(sub)})")

ax.set_title(f"AGDB4 sample locations in {AOI.name} AOI — {len(agdb):,} samples")
ax.set_aspect("equal")
ax.legend(markerscale=4)
plt.tight_layout()

Multi-decade, multi-agency legacy data is exactly what KoBold calls "**dark
data**" when it's scanned reports, and "ingested data" when it's already
standardized like AGDB4 is. This dataset is unusually clean for mineral
exploration — the USGS has already done the painful integration of samples
analyzed by 120 different lab methods into one "best-value" schema with
provenance kept.

In [ ]:
# Read best-value copper from the BV_C_Gd table. AGDB4's column naming:
#   Cu_ppm        numeric best-value in ppm (negatives = censored, e.g. -50 => "<50")
#   Cu_AM         analytical method code (ES_SQ, ICPMS, etc.)
#   Cu_ppm_ALL    all values concatenated as text
import io, zipfile

def load_bv_table(table_name: str) -> pd.DataFrame:
    zip_path = DATA_RAW / "agdb4" / "AGDB4_text.zip"
    with zipfile.ZipFile(zip_path) as zf:
        with zf.open(f"AGDB4_text/{table_name}") as f:
            raw = f.read()
    return pd.read_csv(io.BytesIO(raw), sep=",", low_memory=False, encoding="latin-1")

# Cu is in BV_C_Gd (elements C through Gd); Mo is in BV_Ge_Os.
bv_c_gd = load_bv_table("BV_C_Gd.txt")
bv_ge_os = load_bv_table("BV_Ge_Os.txt")

print(f"BV_C_Gd: {len(bv_c_gd):,} rows")
print(f"  Cu-related cols: {[c for c in bv_c_gd.columns if c.startswith('Cu')]}")
print(f"BV_Ge_Os: {len(bv_ge_os):,} rows")
print(f"  Mo-related cols: {[c for c in bv_ge_os.columns if c.startswith('Mo')]}")

In [ ]:
# Restrict to stream-sediment samples in our AOI, join best-value Cu and Mo.
# Negative Cu values are below-detection sentinels; treat as NaN for now.
aoi_samples = agdb[agdb["PRIMARY_CLASS"] == "sediment"].copy()
joined = aoi_samples.merge(bv_c_gd[["DDPD_ID", "Cu_ppm"]], on="DDPD_ID", how="left")
joined = joined.merge(bv_ge_os[["DDPD_ID", "Mo_ppm"]], on="DDPD_ID", how="left")
joined.loc[joined["Cu_ppm"] < 0, "Cu_ppm"] = np.nan
joined.loc[joined["Mo_ppm"] < 0, "Mo_ppm"] = np.nan
joined = joined.dropna(subset=["Cu_ppm"])
print(f"Sediment samples with Cu assay in AOI: {len(joined):,}")
print(f"Cu range: {joined['Cu_ppm'].min():.1f} to {joined['Cu_ppm'].max():.0f} ppm")
print(f"Cu median: {joined['Cu_ppm'].median():.1f} ppm")

joined_gdf = gpd.GeoDataFrame(
    joined, geometry=gpd.points_from_xy(joined.LONGITUDE, joined.LATITUDE), crs="EPSG:4326"
).to_crs(WORKING_CRS)

fig, ax = plt.subplots(figsize=(10, 5))
# log-scale color, clipped to the middle 95%.
from matplotlib.colors import LogNorm
cu = joined_gdf["Cu_ppm"].clip(lower=1)
vmin, vmax = np.quantile(cu, [0.05, 0.95])
sc = ax.scatter(joined_gdf.geometry.x, joined_gdf.geometry.y,
                c=cu, cmap="inferno", s=6, alpha=0.8,
                norm=LogNorm(vmin=vmin, vmax=vmax))

# Overlay porphyry labels.
por = ardf_porphyry.to_crs(WORKING_CRS)
ax.scatter(por.geometry.x, por.geometry.y, s=80, marker="*",
           facecolor="cyan", edgecolor="black", linewidth=0.7, label="porphyry (ARDF)")

ax.set_title("Stream-sediment Cu (ppm, log scale) + known porphyry sites")
ax.legend(loc="lower left")
ax.set_aspect("equal")
plt.colorbar(sc, ax=ax, label="Cu ppm (log)")
plt.tight_layout()

If the ML model is any good, you should be able to see a weak version of its
job by eye here: brighter (higher-Cu) dots should cluster *around* the cyan
stars. That spatial coincidence is what a classifier will learn — and
what's reassuring about it is that the signal is already visible without
any modeling, it's just noisy and diluted by a lot of "normal" landscape.

## 4. Remote sensing (Sentinel-2)

In [ ]:
# Prefer the per-MGRS-tile mosaic (EastAK v1 approach); fall back to the
# legacy single-reduction composites if only those exist.
s2_path = next(
    (p for p in [
        DATA_RAW / "sentinel2" / f"s2_mosaic_{AOI_KEY}.tif",
        DATA_RAW / "sentinel2" / f"s2_median_{AOI_KEY}.tif",
        DATA_RAW / "sentinel2" / f"s2_mean_{AOI_KEY}.tif",
    ] if p.exists()),
    None,
)
if s2_path is None:
    raise FileNotFoundError(f"No s2_{{median,mean}}_{AOI_KEY}.tif composite found")
s2 = rioxarray.open_rasterio(s2_path, masked=True)
print(f"Loaded {s2_path.name}: shape={s2.shape}, CRS={s2.rio.crs}")
print("Bands (in order):  B02=blue  B03=green  B04=red  B08=NIR  B11=SWIR1  B12=SWIR2")

The Sentinel-2 MSI satellite flies 8 times over this area each summer and
captures reflectance in 13 wavelengths. We pulled the six bands most useful
for mineral mapping. For EastAK the composite is a **per-MGRS-tile mosaic
with multi-scene mean**: for each Sentinel-2 tile footprint we picked the
top-3 lowest-cloud scenes in July–August 2024 (cloud<30%), reprojected
each to 120 m Alaska Albers, and took the per-pixel NaN-skipping mean
before stitching the 19 tiles together.

Why not a single-scene composite or a stackstac median? Three fixable and
one unfixable issue bit us:

1. **`stackstac + dask` hangs on large AOIs** — reproducibly at 40%
   completion regardless of scheduler. Solution: bypass stackstac for
   mosaicking, read scenes directly via rioxarray per MGRS tile.
2. **Planetary Computer SAS tokens expire ~60 min after signing**, so
   long runs hit HTTP 403 on later tiles. Solution: switched to AWS
   Earth Search, which serves the same Sentinel-2 L2A COGs with
   permanent URLs (no signing needed).
3. **Some individual low-cloud scenes are partial acquisitions** that
   only cover part of their MGRS tile. Picking the lowest-cloud scene
   alone left gaps. Solution: mean the top-3 scenes — any pixel only
   needs one scene to have data.
4. **Unfixable**: Rainbow Ridge (MH quadrangle) is a 15 × 15 km patch
   where every available S2 scene in our window has NaN pixels. Real
   data gap, confirmed across both PC and AWS backends with widened
   cloud/date filters. 10 of our 56 porphyry positives fall in this
   gap; downstream modeling inherits NaN S2 features for those training
   points and will document the limitation honestly.

Four tiles at the AOI SE/NE corners had degenerate single-pixel AOI
intersections at 120 m and were dropped (no porphyry positives in
them).

Why those six bands? Different surface materials absorb and reflect light
differently at different wavelengths. Porphyry alteration produces three
spectral fingerprints we can exploit:

- **Iron oxides** (gossan; weathered sulfides): strong absorption in the
  blue (B02), strong reflection in the red (B04). *Iron-oxide index* = B04/B02.
- **Ferrous iron**: reflects more in shortwave infrared (B11) than near
  infrared (B08). *Ferrous index* = B11/B08.
- **Clay + hydroxyl minerals** (kaolinite, illite — common in phyllic
  alteration halos): absorb in B12 more than B11. *Clay index* = B11/B12.

In [ ]:
# Natural-color preview to orient ourselves.
rgb = s2.sel(band=[3, 2, 1])  # B04 red, B03 green, B02 blue
# Downsample for speed (~16M values → ~640k) before percentile + plot.
rgb_preview = rgb[:, ::5, ::5]
p2, p98 = np.nanpercentile(rgb_preview.values, [2, 98])
vis = (rgb_preview.clip(p2, p98) - p2) / (p98 - p2)

fig, ax = plt.subplots(figsize=(10, 5))
vis.plot.imshow(ax=ax, rgb="band")
ax.set_title(f"Sentinel-2 natural color — {AOI.name} AOI (preview, 600 m)")
ax.set_aspect("equal")
plt.tight_layout()

You should see greens and dark reds across most of the image — healthy
boreal forest and tundra. Cooler grey/white patches are exposed bedrock,
landslides, or roads. Bright white near the top is snow on high peaks. This
is the *landscape the classifier will look at* — plus the spectral indices
below.

In [ ]:
# Compute the iron-oxide index: B04 / B02 (on downsampled arrays for speed).
# The feature pipeline will compute this at full 120 m from the raw bands.
red = s2.sel(band=3)[::5, ::5]
blue = s2.sel(band=1)[::5, ::5]
iron_oxide = red / blue

fig, ax = plt.subplots(figsize=(10, 5))
p2, p98 = np.nanpercentile(iron_oxide.values, [2, 98])
iron_oxide.plot(ax=ax, cmap="YlOrRd", vmin=p2, vmax=p98, add_colorbar=True,
                cbar_kwargs={"label": "B04/B02 (iron-oxide index)"})

# Overlay porphyry sites.
por = ardf_porphyry.to_crs(s2.rio.crs)
ax.scatter(por.geometry.x, por.geometry.y, s=80, marker="*",
           facecolor="cyan", edgecolor="black", linewidth=0.7, label="porphyry (ARDF)")

ax.set_title("Iron-oxide index — brighter means more iron-oxide weathering")
ax.legend(loc="lower left")
ax.set_aspect("equal")
plt.tight_layout()

Again: bright pixels near cyan stars are what the model wants to learn. In
practice vegetation masks much of the bedrock signal in Alaska — the feature
NDVI mask will help strip those out.

## 5. Topography (Copernicus GLO-30 DEM)

In [ ]:
dem = rioxarray.open_rasterio(DATA_RAW / "dem" / f"dem_{AOI_KEY}.tif").squeeze()
print(f"DEM shape: {dem.shape}, elevation range {int(dem.min())} – {int(dem.max())} m")

# Downsample for display. At native 30 m, the EastAK DEM is ~98 M pixels —
# matplotlib renders every pixel, which hangs on reasonable hardware. A
# ~1000-wide preview is plenty for a visual sanity check. (The full-res
# data is still used by the downstream feature pipeline.)
dem_preview = dem[::10, ::10]

fig, ax = plt.subplots(figsize=(10, 5))
dem_preview.plot(ax=ax, cmap="terrain", add_colorbar=True,
                 cbar_kwargs={"label": "elevation (m)"})
ax.set_title(f"Copernicus GLO-30 DEM — {AOI.name} (preview, 300 m)")
ax.set_aspect("equal")
plt.tight_layout()

Elevation isn't itself a deposit predictor, but topographic *derivatives*
are. Slope and ruggedness help because porphyry outcrops tend to sit on
uplift blocks that erode faster than surrounding terrane — exposed bedrock,
steeper slopes, less vegetation. The feature pipeline will compute slope + TRI (terrain
ruggedness index) as features.

## 6. Geophysics (magnetic + gravity anomalies)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, label in zip(axes, ["magnetic", "gravity"]):
    p = DATA_RAW / "geophysics" / f"{label}_{AOI_KEY}.tif"
    da = rioxarray.open_rasterio(p, masked=True).squeeze()
    vmin, vmax = np.nanpercentile(da.values, [2, 98])
    da.plot(ax=ax, cmap="RdBu_r", vmin=vmin, vmax=vmax,
            cbar_kwargs={"label": f"{label} anomaly"})

    # Overlay porphyry sites.
    por_proj = ardf_porphyry.to_crs(da.rio.crs)
    ax.scatter(por_proj.geometry.x, por_proj.geometry.y, s=80, marker="*",
               facecolor="yellow", edgecolor="black", linewidth=0.5,
               label="porphyry")

    ax.set_title(f"{label.title()} anomaly with porphyry sites")
    ax.legend(loc="lower left", fontsize=8)
    ax.set_aspect("equal")
plt.tight_layout()

Geophysics lets you "see through" vegetation and thin cover to the rock
below. Two layers:

- **Residual magnetic anomaly** — measures magnetite content. Porphyry
  intrusions often contain magnetite (they're magnetite-series granitoids),
  so they show up as local magnetic highs. A lot of other things do too
  (mafic dikes, ultramafic bodies), so it's never deterministic, just a
  feature.
- **Bouguer gravity** — measures subsurface density. Felsic intrusions are
  *less* dense than surrounding country rock, so they show up as gravity
  *lows*. Again, lots of other things do too.

The magnetic layer we have is USGS national-composite at ~450 m native
pixel, our gravity is ~1.8 km native pixel. Both are coarser than ideal but
reasonable for a regional prospectivity model. A real exploration campaign
would fly high-resolution aeromag over target areas (Fleet Space-style) for
prospect-scale work.

## 7. Geology (USGS SIM 3340)

In [ ]:
geo = gpd.read_file(DATA_RAW / "geology_ak" / f"geology_{AOI_KEY}.gpkg")
print(f"Geology: {len(geo)} polygons in AOI")
print(f"Layer CRS: {geo.crs}")

print("\nTop STATE_SYMBOL codes:")
print(geo["STATE_SYMBOL"].value_counts().head(6).to_string())

print("\nTop age ranges:")
print(geo["AGE_RANGE"].value_counts().head(6).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
# Color by one of the SIM 3340 classification columns.
geo.plot(column="CLASS", ax=ax, cmap="tab20", legend=False, alpha=0.7, edgecolor="none")

# Overlay porphyry sites.
por_proj = ardf_porphyry.to_crs(geo.crs)
ax.scatter(por_proj.geometry.x, por_proj.geometry.y, s=80, marker="*",
           facecolor="red", edgecolor="black", linewidth=0.7, label="porphyry (ARDF)")

ax.set_title(f"SIM 3340 geologic units (N={len(geo)}) + porphyry sites")
ax.legend(loc="lower left")
ax.set_aspect("equal")
plt.tight_layout()

Notice how the porphyry stars sit preferentially on particular polygons —
late Cretaceous intrusive rocks. That's geology-as-feature in action: the
rock unit a pixel is on is a strong predictor because porphyry *only* forms
in specific geological settings. The `CLASS` and `STATE_SYMBOL` columns are
numeric codes that join to the `nsaunits` / `nsakey` tables in the GDB for
human-readable unit names — the feature pipeline resolves those and builds a one-hot
encoding of the top lithologic units as features.

## 8. Putting it together — the multi-layer view

In [ ]:
# One plot showing DEM background + geology outline + geochem overlay + porphyry stars.
fig, ax = plt.subplots(figsize=(12, 6))

# Layer 1: DEM hillshade-ish background (downsampled for plot speed).
dem_preview.plot(ax=ax, cmap="Greys_r", alpha=0.8, add_colorbar=False)

# Layer 2: sediment Cu sample points, warm color scale.
from matplotlib.colors import LogNorm
cu = joined_gdf["Cu_ppm"].clip(lower=1)
vmin, vmax = np.quantile(cu, [0.1, 0.95])
sc = ax.scatter(joined_gdf.geometry.x, joined_gdf.geometry.y,
                c=cu, cmap="YlOrRd", s=8, alpha=0.9,
                norm=LogNorm(vmin=vmin, vmax=vmax))
plt.colorbar(sc, ax=ax, label="sediment Cu (ppm, log)")

# Layer 3: porphyry sites.
ax.scatter(por_proj.geometry.x, por_proj.geometry.y, s=120, marker="*",
           facecolor="lime", edgecolor="black", linewidth=0.8,
           label=f"ARDF porphyry (N={len(ardf_porphyry)})")

ax.set_title(f"{AOI.name} composite — DEM + stream-sediment Cu + porphyry labels")
ax.legend(loc="lower left")
ax.set_aspect("equal")
plt.tight_layout()

This is what the ML pipeline consumes, sampled per pixel.
Every pixel will have:

- ~30 geochemical aggregates (Cu, Mo, Au, Ag, As, Sb, Pb, Zn, Bi, Te, each
  as mean + max + sample-count within a 5-km neighborhood)
- 3 Sentinel-2 alteration indices (iron-oxide, ferrous, clay)
- 3 geophysics values (magnetic anomaly, Bouguer gravity, gravity HGM)
- 1 lithology class (one-hot of top-N most common units)
- 1 fault-proximity distance
- 3 topographic derivatives (elevation, slope, TRI)

That's about 40 features per pixel. The classifier learns which combinations
correlate with the porphyry label — then we ask it to score every pixel in
the AOI and produce the prospectivity map.

## Key takeaways before modeling

1. **The label problem is stark.** 62 known porphyries across ~67,000 km²
   (millions of pixels). Still rare in absolute terms —
   random-undersampling vs PU learning is a real choice.
2. **Multi-modal is the whole point.** No single layer predicts deposits
   well; the signal lives in *how geochemistry, geophysics, and remote
   sensing correlate*. This is why this problem is AI-shaped and not
   classical-statistics-shaped.
3. **The data are messy but honest.** Legacy samples from four decades,
   multiple agencies, multiple analytical methods. AGDB4's "best value"
   schema gets us most of the way; USGS has already done the painful
   harmonization.
4. **The AOI has structure.** Porphyry sites cluster geographically —
   along specific geologic units and sub-regions of the
   Wrangellia–Yukon-Tanana boundary. A model that respects spatial
   autocorrelation (which is why we use spatial block CV) will report
   honest performance.

Next up: [`baseline_model.qmd`](baseline_model.qmd) builds the analysis grid, computes the features above, and
trains a first logistic-regression + Random Forest pair with spatial
cross-validation.